In [1]:
# Use pandas to load and summarize the datasets as tables.
import pandas as pd
# Use SciPy for distribution-shape statistics such as skewness and kurtosis.
from scipy import stats
# Use Matplotlib to visualize distributions and spread.
import matplotlib.pyplot as plt

import seaborn as sns 

In [ ]:
# Request the USGS earthquake catalog for 2024, keeping events of magnitude 4.5 or greater.
url = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2024-01-01&endtime=2024-12-31&minmagnitude=4.5"
# Read the returned CSV directly into a DataFrame for analysis.
eq_df = pd.read_csv(url)

In [ ]:
# Identify the dataset and show how many matching earthquakes were returned.
print("USGS Earthquake Data (Magnitude >= 4.5, 2024)")
print(f"Total records: {len(eq_df)}")

In [ ]:
# Preview key event details: when and where each earthquake occurred, its depth, and magnitude.
print(eq_df[['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'place']].head(10))
# Print a blank line to separate this preview from the next notebook output.
print()

In [ ]:
# Compare how earthquake magnitude and depth vary across the selected events.
print("=" * 120)
print("DISPERSION ANALYSIS: Earthquake Magnitude & Depth")
print("=" * 120)

# Store the two measurements together so the same summary calculations can be reused.
variables = {
    'Magnitude (mag)': eq_df['mag'],
    'Depth (km)': eq_df['depth']
}

stats_results = []
for label, data in variables.items():
    # Exclude missing observations before calculating statistics.
    data_clean = data.dropna()
    stats_results.append({
        'Variable': label,
        'Count': len(data_clean),
        # Mean and median describe the center; range, variance, and SD measure spread.
        'Mean': round(data_clean.mean(), 3),
        'Median': round(data_clean.median(), 3),
        'Range': round(data_clean.max() - data_clean.min(), 3),
        'Variance': round(data_clean.var(), 3),
        'Std Dev': round(data_clean.std(), 3),
        # Quartiles and IQR describe the middle half of the observations.
        'IQR': round(data_clean.quantile(0.75) - data_clean.quantile(0.25), 3),
        'Q1 (25%)': round(data_clean.quantile(0.25), 3),
        'Q3 (75%)': round(data_clean.quantile(0.75), 3),
        'Min': round(data_clean.min(), 3),
        'Max': round(data_clean.max(), 3),
        # CV expresses standard deviation relative to the mean, as a percentage.
        'CV (%)': round((data_clean.std() / data_clean.mean()) * 100, 3),
        # Skewness and kurtosis describe distribution shape and tail weight.
        'Skewness': round(stats.skew(data_clean), 3),
        'Kurtosis': round(stats.kurtosis(data_clean), 3)
    })

# Display both variables' results side by side in a compact table.
stats_df = pd.DataFrame(stats_results)
print(stats_df.to_string(index=False))
print()

In [ ]:
# Use pandas' built-in summary as a quick cross-check of the main earthquake measurements.
print("-" * 30)
print("Pandas .describe() output:")
print("-" * 30)
# describe() reports count, mean, standard deviation, quartiles, and extremes.
print(eq_df[['mag', 'depth']].describe().round(3))
print()

In [ ]:
# Arrange box plots and histograms in a 2-by-2 figure for direct comparison.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dispersion Analysis: USGS Earthquake Data (2024, Mag >= 4.5)', fontsize=14, fontweight='bold')

# A box plot summarizes the magnitude median, quartiles, and potential outliers.
ax1 = axes[0, 0]
bp1 = ax1.boxplot(eq_df['mag'].dropna(), patch_artist=True, vert=True)
bp1['boxes'][0].set_facecolor('#2196F3')
bp1['boxes'][0].set_alpha(0.7)
ax1.set_ylabel('Magnitude')
ax1.set_title('Box Plot: Earthquake Magnitude')
ax1.grid(axis='y', alpha=0.3)
# Mark the first and third quartiles and the interquartile range (IQR).
q1_m, q3_m = eq_df['mag'].quantile([0.25, 0.75])
iqr_m = q3_m - q1_m
ax1.annotate(f'IQR = {iqr_m:.2f}\nQ1={q1_m:.2f}, Q3={q3_m:.2f}', xy=(1.3, q1_m), fontsize=9)

# A second box plot shows the spread and outliers in earthquake depth.
ax2 = axes[0, 1]
bp2 = ax2.boxplot(eq_df['depth'].dropna(), patch_artist=True, vert=True)
bp2['boxes'][0].set_facecolor('#FF5722')
bp2['boxes'][0].set_alpha(0.7)
ax2.set_ylabel('Depth (km)')
ax2.set_title('Box Plot: Earthquake Depth')
ax2.grid(axis='y', alpha=0.3)
# Annotate the middle half of the depth observations using its quartiles.
q1_d, q3_d = eq_df['depth'].quantile([0.25, 0.75])
iqr_d = q3_d - q1_d
ax2.annotate(f'IQR = {iqr_d:.2f}\nQ1={q1_d:.2f}, Q3={q3_d:.2f}', xy=(1.3, q1_d), fontsize=9)

# The magnitude histogram shows frequency; vertical markers indicate the mean and +/- 1 SD.
ax3 = axes[1, 0]
ax3.hist(eq_df['mag'].dropna(), bins=40, color='#2196F3', edgecolor='white', alpha=0.8)
ax3.axvline(eq_df['mag'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean = {eq_df['mag'].mean():.2f}")
ax3.axvline(eq_df['mag'].mean() + eq_df['mag'].std(), color='orange', linestyle=':', linewidth=2, label=f"+1 SD = {eq_df['mag'].std():.2f}")
ax3.axvline(eq_df['mag'].mean() - eq_df['mag'].std(), color='orange', linestyle=':', linewidth=2, label=f"-1 SD")
ax3.set_xlabel('Magnitude')
ax3.set_ylabel('Frequency')
ax3.set_title(f'Histogram: Magnitude (Std Dev = {eq_df["mag"].std():.3f})')
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

# The depth histogram uses the same mean and standard-deviation reference lines.
ax4 = axes[1, 1]
ax4.hist(eq_df['depth'].dropna(), bins=40, color='#FF5722', edgecolor='white', alpha=0.8)
ax4.axvline(eq_df['depth'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean = {eq_df['depth'].mean():.1f}")
ax4.axvline(eq_df['depth'].mean() + eq_df['depth'].std(), color='blue', linestyle=':', linewidth=2, label=f"+1 SD = {eq_df['depth'].std():.1f}")
ax4.axvline(eq_df['depth'].mean() - eq_df['depth'].std(), color='blue', linestyle=':', linewidth=2, label=f"-1 SD")
ax4.set_xlabel('Depth (km)')
ax4.set_ylabel('Frequency')
ax4.set_title(f'Histogram: Depth (Std Dev = {eq_df["depth"].std():.3f})')
ax4.legend(fontsize=9)
ax4.grid(axis='y', alpha=0.3)

# Adjust spacing so titles and labels fit, then display the figure.
plt.tight_layout()
plt.gca()

In [ ]:
# Load a second geoscience dataset: a global catalog of volcano locations and elevations.
# Catalog source: https://raw.githubusercontent.com/plotly/datasets/master/volcano_db.csv
volcano_url = "https://raw.githubusercontent.com/plotly/datasets/master/volcano_db.csv"
# The catalog contains some legacy Latin-1 characters in text fields.
volcano_df = pd.read_csv(volcano_url, encoding="latin-1")
# Convert elevation to numbers and omit rows without a usable elevation value.
volcano_elev = pd.to_numeric(volcano_df["Elev"], errors="coerce").dropna()

# Report the number of records and clarify that elevation is topography, not eruption strength.
print("VOLCANO ELEVATION DISPERSION")
print(f"Catalog records: {len(volcano_df)} | usable elevations: {len(volcano_elev)}")
print("Elevation is measured in meters; this describes volcano topography, not eruption intensity.")
# Summarize the center, total spread, middle-half spread, and relative variability.
volcano_summary = pd.Series({
    "Mean (m)": volcano_elev.mean(),
    "Median (m)": volcano_elev.median(),
    "Range (m)": volcano_elev.max() - volcano_elev.min(),
    "Variance (m^2)": volcano_elev.var(),
    "Std Dev (m)": volcano_elev.std(),
    "IQR (m)": volcano_elev.quantile(0.75) - volcano_elev.quantile(0.25),
    # Coefficient of variation compares the standard deviation with the mean.
    "CV (%)": volcano_elev.std() / volcano_elev.mean() * 100,
}).round(2)
print(volcano_summary.to_string())

# Reuse the mean, standard deviation, and quartiles as plot reference values.
volcano_mean = volcano_elev.mean()
volcano_std = volcano_elev.std()
volcano_q1, volcano_q3 = volcano_elev.quantile([0.25, 0.75])



In [ ]:
# visualize the volcano elevation distribution and spread using a histogram and box plot.
fig_volcano, volcano_axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig_volcano.suptitle("Dispersion Analysis: Global Volcano Elevations", fontweight="bold")

# The histogram shows the elevation distribution with its mean and +/- 1 SD marked.
volcano_axes[0].hist(volcano_elev, bins=35, color="#2a7f62", edgecolor="white", alpha=0.85)
volcano_axes[0].axvline(volcano_mean, color="#c43c39", linestyle="--", label=f"Mean = {volcano_mean:.0f} m")
volcano_axes[0].axvline(volcano_mean - volcano_std, color="#d18b26", linestyle=":", label="Mean +/- 1 SD")
volcano_axes[0].axvline(volcano_mean + volcano_std, color="#d18b26", linestyle=":")
volcano_axes[0].set(title="Elevation distribution", xlabel="Elevation (m)", ylabel="Volcano count")
volcano_axes[0].legend()
volcano_axes[0].grid(axis="y", alpha=0.25)

# The box plot highlights the median, IQR, and unusually high or low elevations.
volcano_axes[1].boxplot(volcano_elev, patch_artist=True, tick_labels=["Volcanoes"])
volcano_axes[1].set(title="Elevation spread and outliers", ylabel="Elevation (m)")
volcano_axes[1].annotate(f"IQR = {volcano_q3 - volcano_q1:.0f} m", xy=(1.15, volcano_q1))
volcano_axes[1].grid(axis="y", alpha=0.25)

# Keep the two plots neatly spaced before displaying them.
plt.tight_layout()
plt.show()

In [ ]:
# Compare annual oil production across countries using Our World in Data's grapher export.
# The underlying source combines Energy Institute, EIA, and historical estimates.
# The oil measure includes crude oil, condensates, natural gas liquids, and other liquid fuels.
oil_url = "https://ourworldindata.org/grapher/oil-production-by-country.csv"
oil_df = pd.read_csv(oil_url)
# Select the most recent year available in the downloaded dataset.
latest_oil_year = int(oil_df["Year"].max())
# Keep country rows with ISO-style three-letter codes, excluding regional aggregates.
latest_oil = oil_df.loc[
    (oil_df["Year"] == latest_oil_year)
    & oil_df["Code"].notna()
    & (oil_df["Code"].str.len() == 3)
].copy()
# Ensure production is numeric, then exclude missing and zero-production countries.
latest_oil["Oil"] = pd.to_numeric(latest_oil["Oil"], errors="coerce")
oil_producers = latest_oil.loc[latest_oil["Oil"] > 0, "Oil"].dropna()

# Explain which year, population, units, and source the following summary represents.
print(f"OIL PRODUCTION DISPERSION BY COUNTRY ({latest_oil_year})")
print(f"Producing countries with data: {len(oil_producers)}")
print("Measure: terawatt-hours (TWh); non-producing countries are excluded.")
print("Source: Our World in Data, based on Energy Institute, EIA, and historical estimates.")
# Quantify typical production and how widely production differs between countries.
oil_summary = pd.Series({
    "Mean (TWh)": oil_producers.mean(),
    "Median (TWh)": oil_producers.median(),
    "Range (TWh)": oil_producers.max() - oil_producers.min(),
    "Variance (TWh^2)": oil_producers.var(),
    "Std Dev (TWh)": oil_producers.std(),
    "IQR (TWh)": oil_producers.quantile(0.75) - oil_producers.quantile(0.25),
    # CV expresses standard deviation as a percentage of the mean.
    "CV (%)": oil_producers.std() / oil_producers.mean() * 100,
}).round(2)
print(oil_summary.to_string())

# NumPy creates logarithmically spaced bins because production is highly right-skewed.
import numpy as np

# Store center and quartile values for reference lines and the IQR annotation.
oil_mean = oil_producers.mean()
oil_median = oil_producers.median()
oil_q1, oil_q3 = oil_producers.quantile([0.25, 0.75])
# Log-spaced bins preserve detail among smaller producers while covering the full range.
oil_bins = np.logspace(
    np.floor(np.log10(oil_producers.min())),
    np.ceil(np.log10(oil_producers.max())),
    30,
)


In [ ]:
# Visualize the distribution and spread of country oil production using a histogram and box plot.
fig_oil, oil_axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig_oil.suptitle(f"Dispersion Analysis: Country Oil Production ({latest_oil_year})", fontweight="bold")

# Show country counts across production levels; the log x-axis makes small and large values visible.
oil_axes[0].hist(oil_producers, bins=oil_bins, color="#b87932", edgecolor="white", alpha=0.85)
oil_axes[0].set_xscale("log")
# Compare the mean with the median to make the distribution's skew apparent.
oil_axes[0].axvline(oil_mean, color="#c43c39", linestyle="--", label=f"Mean = {oil_mean:,.0f} TWh")
oil_axes[0].axvline(oil_median, color="#176b67", linestyle="-.", label=f"Median = {oil_median:,.0f} TWh")
oil_axes[0].set(title="Production distribution (log scale)", xlabel="Annual oil production (TWh)", ylabel="Country count")
oil_axes[0].legend()
oil_axes[0].grid(axis="y", alpha=0.25)

# A log-scale box plot shows the middle spread and high-production outliers.
oil_axes[1].boxplot(oil_producers, patch_artist=True, tick_labels=["Countries"])
oil_axes[1].set_yscale("log")
oil_axes[1].set(title="Spread and high-production outliers (log scale)", ylabel="Annual oil production (TWh)")
oil_axes[1].annotate(f"IQR = {oil_q3 - oil_q1:,.0f} TWh", xy=(1.12, oil_q3))
oil_axes[1].grid(axis="y", alpha=0.25)

# Adjust spacing and display the oil-production dispersion plots.
plt.tight_layout()
plt.show()

In [ ]:
# List the datasets available through Seaborn's public helper.
# for data in sns.get_dataset_names():
#     print(data)

In [ ]:
# Load Seaborn's Old Faithful geyser dataset, which includes eruption duration and waiting time.
# The numeric measurements are in minutes; "kind" identifies the eruption category.
geyser_df = sns.load_dataset("geyser")
print("OLD FAITHFUL GEYSER DISPERSION ANALYSIS")
print(f"Observations: {len(geyser_df)}")
print("Eruption categories:")
print(geyser_df["kind"].value_counts().sort_index().to_string())

# Analyze both numerical variables with the same descriptive statistics used above.
geyser_variables = {
    "Eruption duration (min)": geyser_df["duration"],
    "Waiting time (min)": geyser_df["waiting"],
}
geyser_results = []
for label, values in geyser_variables.items():
    # Exclude missing measurements before calculating statistics.
    values_clean = values.dropna()
    geyser_results.append({
        "Variable": label,
        "Count": len(values_clean),
        "Mean": round(values_clean.mean(), 3),
        "Median": round(values_clean.median(), 3),
        "Range": round(values_clean.max() - values_clean.min(), 3),
        "Variance": round(values_clean.var(), 3),
        "Std Dev": round(values_clean.std(), 3),
        "IQR": round(values_clean.quantile(0.75) - values_clean.quantile(0.25), 3),
        "Q1 (25%)": round(values_clean.quantile(0.25), 3),
        "Q3 (75%)": round(values_clean.quantile(0.75), 3),
        "Min": round(values_clean.min(), 3),
        "Max": round(values_clean.max(), 3),
        "CV (%)": round(values_clean.std() / values_clean.mean() * 100, 3),
        "Skewness": round(stats.skew(values_clean), 3),
        "Kurtosis": round(stats.kurtosis(values_clean), 3),
    })

# Present the two measurements together so their scales and spreads are easy to compare.
geyser_stats_df = pd.DataFrame(geyser_results)
print("\nDispersion statistics:")
print(geyser_stats_df.to_string(index=False))
print("\nPandas describe() cross-check:")
print(geyser_df[["duration", "waiting"]].describe().round(3))

# Prepare means, standard deviations, and quartiles for the reference lines and IQR labels.
duration = geyser_df["duration"].dropna()
waiting = geyser_df["waiting"].dropna()
duration_mean, duration_std = duration.mean(), duration.std()
waiting_mean, waiting_std = waiting.mean(), waiting.std()
duration_q1, duration_q3 = duration.quantile([0.25, 0.75])
waiting_q1, waiting_q3 = waiting.quantile([0.25, 0.75])



In [ ]:
# Use matching box plots and histograms to show quartiles, outliers, frequency, and spread.
geyser_fig, geyser_axes = plt.subplots(2, 2, figsize=(14, 10))
geyser_fig.suptitle("Dispersion Analysis: Old Faithful Geyser", fontsize=14, fontweight="bold")

# The duration box plot marks the middle 50% with its box and potential outliers beyond the whiskers.
geyser_axes[0, 0].boxplot(duration, patch_artist=True, orientation="vertical")
geyser_axes[0, 0].patches[0].set_facecolor("#168b83")
geyser_axes[0, 0].set(title="Eruption duration", ylabel="Duration (min)", xticks=[])
geyser_axes[0, 0].annotate(f"IQR = {duration_q3 - duration_q1:.2f} min", xy=(1.12, duration_q1))
geyser_axes[0, 0].grid(axis="y", alpha=0.25)

# The waiting-time box plot summarizes the spread between eruptions in the same way.
geyser_axes[0, 1].boxplot(waiting, patch_artist=True, orientation="vertical")
geyser_axes[0, 1].patches[0].set_facecolor("#d67a36")
geyser_axes[0, 1].set(title="Waiting time", ylabel="Waiting time (min)", xticks=[])
geyser_axes[0, 1].annotate(f"IQR = {waiting_q3 - waiting_q1:.2f} min", xy=(1.12, waiting_q1))
geyser_axes[0, 1].grid(axis="y", alpha=0.25)

# The duration histogram shows the distribution around its mean and one standard deviation.
geyser_axes[1, 0].hist(duration, bins=25, color="#168b83", edgecolor="white", alpha=0.85)
geyser_axes[1, 0].axvline(duration_mean, color="#c43c39", linestyle="--", label=f"Mean = {duration_mean:.2f} min")
geyser_axes[1, 0].axvline(duration_mean - duration_std, color="#d18b26", linestyle=":", label="Mean +/- 1 SD")
geyser_axes[1, 0].axvline(duration_mean + duration_std, color="#d18b26", linestyle=":")
geyser_axes[1, 0].set(title="Eruption duration distribution", xlabel="Duration (min)", ylabel="Frequency")
geyser_axes[1, 0].legend()
geyser_axes[1, 0].grid(axis="y", alpha=0.25)

# The waiting-time histogram uses the same references to make the two distributions comparable.
geyser_axes[1, 1].hist(waiting, bins=25, color="#d67a36", edgecolor="white", alpha=0.85)
geyser_axes[1, 1].axvline(waiting_mean, color="#c43c39", linestyle="--", label=f"Mean = {waiting_mean:.2f} min")
geyser_axes[1, 1].axvline(waiting_mean - waiting_std, color="#176b67", linestyle=":", label="Mean +/- 1 SD")
geyser_axes[1, 1].axvline(waiting_mean + waiting_std, color="#176b67", linestyle=":")
geyser_axes[1, 1].set(title="Waiting time distribution", xlabel="Waiting time (min)", ylabel="Frequency")
geyser_axes[1, 1].legend()
geyser_axes[1, 1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
import statsmodels as sm

In [ ]:
# Load the classic annual Nile flow series measured at Aswan from 1871 through 1970.
# The source reports annual flow in units of 10^8 cubic meters.
nile_url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/Nile.csv"
nile_df = pd.read_csv(nile_url).rename(columns={"time": "Year", "value": "Annual flow (10^8 m^3)"})
nile_df = nile_df[["Year", "Annual flow (10^8 m^3)"]]
nile_flow = nile_df["Annual flow (10^8 m^3)"].dropna()

print("NILE RIVER FLOW DISPERSION AT ASWAN (1871-1970)")
print(f"Annual observations: {len(nile_flow)}")
print("Measurement: annual flow in 10^8 cubic meters; this is volume per year, not m^3/s.")

# Calculate center, absolute spread, relative spread, and distribution shape.
nile_summary = pd.Series({
    "Mean": nile_flow.mean(),
    "Median": nile_flow.median(),
    "Range": nile_flow.max() - nile_flow.min(),
    "Variance": nile_flow.var(),
    "Std Dev": nile_flow.std(),
    "IQR": nile_flow.quantile(0.75) - nile_flow.quantile(0.25),
    "Q1 (25%)": nile_flow.quantile(0.25),
    "Q3 (75%)": nile_flow.quantile(0.75),
    "Min": nile_flow.min(),
    "Max": nile_flow.max(),
    "CV (%)": nile_flow.std() / nile_flow.mean() * 100,
    "Skewness": stats.skew(nile_flow),
    "Kurtosis": stats.kurtosis(nile_flow),
}).round(3)
print(nile_summary.to_string())


In [ ]:
# Calculate reference values for the box plot and histogram.
nile_mean = nile_flow.mean()
nile_std = nile_flow.std()
nile_q1, nile_q3 = nile_flow.quantile([0.25, 0.75])
nile_fig, nile_axes = plt.subplots(1, 3, figsize=(16, 4.5))
nile_fig.suptitle("Dispersion Analysis: Nile River Annual Flow at Aswan", fontweight="bold")

# The box plot summarizes the median, middle 50%, and potential outliers.
nile_axes[0].boxplot(nile_flow, patch_artist=True, tick_labels=["Annual flow"])
nile_axes[0].patches[0].set_facecolor("#268b85")
nile_axes[0].set(title="Annual flow spread", ylabel="Annual flow (10^8 m^3)")
nile_axes[0].annotate(f"IQR = {nile_q3 - nile_q1:.1f}", xy=(1.12, nile_q1))
nile_axes[0].grid(axis="y", alpha=0.25)

# The histogram shows the distribution with the mean and one standard deviation marked.
nile_axes[1].hist(nile_flow, bins=18, color="#268b85", edgecolor="white", alpha=0.85)
nile_axes[1].axvline(nile_mean, color="#c43c39", linestyle="--", label=f"Mean = {nile_mean:.0f}")
nile_axes[1].axvline(nile_mean - nile_std, color="#d18b26", linestyle=":", label="Mean +/- 1 SD")
nile_axes[1].axvline(nile_mean + nile_std, color="#d18b26", linestyle=":")
nile_axes[1].set(title="Distribution of annual flow", xlabel="Annual flow (10^8 m^3)", ylabel="Year count")
nile_axes[1].legend()
nile_axes[1].grid(axis="y", alpha=0.25)

# The time-series plot preserves the order of years and shows long-term changes.
nile_axes[2].plot(nile_df["Year"], nile_flow, color="#2878a0", linewidth=1.3)
nile_axes[2].set(title="Annual flow by year", xlabel="Year", ylabel="Annual flow (10^8 m^3)")
nile_axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Statistical Concepts

This notebook uses descriptive statistics to summarize and compare variation in Nile River annual flow at Aswan, Blue Nile discharge, earthquake measurements, volcano elevations, country-level oil production, and Old Faithful geyser eruption durations and waiting times.

### Center and Spread
- **Mean:** arithmetic average; it is sensitive to unusually high or low values.
- **Median:** middle value; it is less affected by extreme observations than the mean.
- **Range:** maximum minus minimum, showing the full observed spread.
- **Variance:** average squared deviation from the mean. Its units are squared (for example, km^2 or TWh^2).
- **Standard deviation (SD):** square root of the variance, expressed in the original measurement units. The plots mark the mean and one SD on either side.
- **Quartiles and IQR:** Q1 is the 25th percentile, Q3 is the 75th percentile, and the interquartile range (IQR = Q3 - Q1) covers the middle 50% of values. It is less affected by extremes than the range.
- **Coefficient of variation (CV):** SD divided by the mean, expressed as a percentage. It can compare relative spread for positive, ratio-scale measurements, but should be interpreted cautiously for logarithmic earthquake magnitude and elevations that can be below sea level.

### Distribution Shape and Plots
- **Skewness** describes distribution asymmetry; a long right tail often pulls the mean above the median.
- **Kurtosis** summarizes tail weight relative to a normal distribution and can help flag extreme observations.
- **Histograms** show the frequency and shape of a distribution. **Box plots** show the median and quartiles; their whiskers extend to the most extreme points within 1.5 IQR of the quartiles, with more distant points plotted as potential outliers.
- The geyser example analyzes eruption duration and waiting time in minutes. Its duration histogram has distinct short and long clusters, so a single mean and standard deviation do not show the full distribution structure.
- The Blue Nile example compares 12 monthly mean discharges at Roseires Dam. Its range, SD, and IQR describe seasonal differences among months, not year-to-year variability.
- The Nile-at-Aswan example uses annual flow measurements from 1871 to 1970 in units of 10^8 m^3. Its time-series plot preserves year order; full-period dispersion statistics alone can hide changes over time.
- The oil-production plots use a **logarithmic axis** because values span several orders of magnitude. Equal visual steps represent multiplicative, not additive, changes; the underlying production values and summary statistics remain unchanged.










 > Please take note that i have used AI to develop this notebook.